<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_05_report.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_06 · Notebook 05 — The Report

**Paired with lecture block L6 · Part 1**

The last notebook of Part 1. What is marked is not whether your numbers match
anyone else's — it is whether you can say what you measured, what it means, and
where it stops being true.

## How to use this notebook

Run section 0 to pull in what notebooks 01 to 04 saved. Then replace every
string in section 2. Section 3 assembles the whole thing into a markdown file
you submit.

Word limits are enforced by the check in section 3, and they are tight on
purpose. An answer that needs six hundred words has not been understood yet.

## 0 · Load what the other notebooks produced

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_6_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex06-training-lab/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import os
import numpy as np
import Ex_6_core as core

paths = {
    "01 · losses":       "nb01_losses.npz",
    "02 · optimisers":   "nb02_optimisers.npz",
    "03 · transfer":     "nb03_transfer.npz",
    "04 · quantisation": "nb04_quantisation.npz",
}

R = {}
for label, fname in paths.items():
    full = os.path.join(core.OUTPUT_DIR, fname)
    if os.path.exists(full):
        R[label] = np.load(full, allow_pickle=True)
        print(f"  loaded  {label}")
    else:
        print(f"  MISSING {label}  ({fname}) -- run that notebook first")

if len(R) < len(paths):
    print("\nSome results are missing. The report will have gaps.")

---

## 0b · Your personal seed

Every notebook in this exercise set fixes the seed to 0, so the printed
"what you should see" blocks are true on any machine. That is right for
checking your work and wrong for reporting it — with one seed the whole cohort
produces identical numbers.

So the numbers below are **yours**. Put your study number in, run the cell, and
quote what it prints in your answers where the questions ask for it. Your
supervisor can regenerate exactly these numbers from your study number alone,
so they are worth getting right and pointless to invent.

In [ ]:
STUDENT_NUMBER = "20241234"        # <- your AAU study number

SEED = core.personal_seed(STUDENT_NUMBER)
print("study number :", STUDENT_NUMBER)
print("your seed    :", SEED)

# Your own calibration run on the same load cell.
x_you, y_you = core.calibration_dataset(n=40, seed=SEED)
design = np.stack([x_you, np.ones_like(x_you)], axis=1)
A_YOU, B_YOU = np.linalg.lstsq(design, y_you, rcond=None)[0]
SIGMA_YOU = float(np.sqrt(np.mean((y_you - (A_YOU * x_you + B_YOU)) ** 2)))

print()
print(f"  your fitted slope     : {A_YOU:.4f}   (true {core.TRUE_SLOPE})")
print(f"  your fitted intercept : {B_YOU:.4f}   (true {core.TRUE_INTERCEPT})")
print(f"  your residual sigma   : {SIGMA_YOU:.4f}   (true {core.TRUE_SIGMA})")

## 1 · The evidence, gathered

Read these off before writing anything. Several of the questions below are
about numbers you may not have looked at since you produced them.

In [ ]:
if "01 · losses" in R:
    d = R["01 · losses"]
    print("NOTEBOOK 01")
    print(f"  least-squares slope/intercept : {d['a_ls']:.4f} / {d['b_ls']:.4f}")
    print(f"  sigma from residuals          : {d['sigma_hat']:.4f}"
          f"   (true {core.TRUE_SIGMA})")
    print(f"  cross entropy, numpy vs torch : {abs(d['ce_numpy']-d['ce_torch']):.2e}")
    print(f"  accuracy   CE / MSE           : {d['acc_ce']:.3f} / {d['acc_mse']:.3f}")
    print(f"  held-out CE, CE / MSE trained : {d['heldout_ce_ce']:.4f} / "
          f"{d['heldout_ce_mse']:.4f}")
    print(f"  line fit under MSE / MAE      : {d['fit_mse'][0]:.3f} / "
          f"{d['fit_mae'][0]:.3f}   (true {core.TRUE_SLOPE})")

if "02 · optimisers" in R:
    d = R["02 · optimisers"]
    print("\nNOTEBOOK 02")
    for n, v in zip(d["final_names"], d["final_losses"]):
        print(f"  {str(n):<22s} {v:.3e}")

if "03 · transfer" in R:
    d = R["03 · transfer"]
    print("\nNOTEBOOK 03")
    print(f"  source model on A / on B : {d['acc_source_A']:.3f} / "
          f"{d['acc_source_B']:.3f}")
    for n, v in zip(d["strategy_names"], d["strategy_acc"]):
        print(f"  {str(n):<28s} {v:.3f}")

if "04 · quantisation" in R:
    d = R["04 · quantisation"]
    print("\nNOTEBOOK 04")
    print(f"  size  float32 / int8 : {int(d['base_bytes']):,} / "
          f"{int(d['quant_bytes']):,} bytes"
          f"   ({d['base_bytes']/d['quant_bytes']:.2f}x)")
    print(f"  latency ratio        : "
          f"{d['base_seconds']/d['quant_seconds']:.2f}x")
    print(f"  accuracy delta       : "
          f"{d['quant_accuracy']-d['base_accuracy']:+.3f}")

## 2 · Your answers

Replace every string. Keep to the word limits.

In [ ]:
# TODO: write your report. Every string below must be replaced.

Q1_LOSS_FROM_NOISE = """
(120 words) Notebook 01 showed that least squares is what you get by assuming
Gaussian noise and maximising the likelihood. State the assumption precisely,
then name one measurement situation in your own field where it is wrong, and
say which loss the correct noise assumption would give you instead.
"""

Q2_OPTIMISER_HANDOFF = """
(120 words) Every PINN in Part 2 is trained with Adam and then L-BFGS. Using
your numbers from notebook 02, explain what each contributes. Then state one
thing a very low final loss does NOT establish.
"""

Q3_LEARNING_RATE = """
(100 words) You have two failed training runs. One ends with a loss of nan; the
other converges smoothly to a value far above what the model should reach.
Explain what has gone wrong in each and how you would tell them apart from the
training log alone.
"""

Q4_TRANSFER = """
(150 words) Report the four accuracies from notebook 03 and say which strategy
you would take to the workshop, with reasons. Include the label budget at
which your answer would change, and say how you know.
"""

Q5_DEPLOYMENT = """
(150 words) Report the three deployment numbers from notebook 04. If your
latency did not improve, explain why and do not apologise for it. Then answer
the question the notebook raised: was quantisation the right lever for this
model, or would a narrower float model have been better?
"""

Q6_WHAT_YOU_DISTRUST = """
(120 words) Name the result from Ex_06 you trust least, and say exactly what
experiment would settle it. Answers naming a specific number and a specific
test score higher than general statements about needing more data.
"""

NAME = "your name"
GROUP = "your group"

raise NotImplementedError("Write your report, then delete this line")

## 3 · Check, assemble, save

The check counts words and refuses anything left at its default. Fix what it
reports, then run it again.

In [ ]:
answers = {
    "1 · Where the loss comes from": (Q1_LOSS_FROM_NOISE, 120),
    "2 · Adam, then L-BFGS": (Q2_OPTIMISER_HANDOFF, 120),
    "3 · Two ways a learning rate fails": (Q3_LEARNING_RATE, 100),
    "4 · A second machine, thirty labels": (Q4_TRANSFER, 150),
    "5 · Making it fit on the device": (Q5_DEPLOYMENT, 150),
    "6 · What you distrust": (Q6_WHAT_YOU_DISTRUST, 120),
}

problems = []
for title, (text, limit) in answers.items():
    words = len(text.split())
    if text.strip().startswith("(") or "(%d words)" % limit in text:
        problems.append(f"{title}: still the prompt")
    elif words > limit * 1.15:
        problems.append(f"{title}: {words} words, limit {limit}")
    elif words < limit * 0.4:
        problems.append(f"{title}: {words} words, too short")

if problems:
    print("Not ready:")
    for p in problems:
        print("  -", p)
else:
    lines = [f"# Ex_06 — Training Lab", "",
             f"**{NAME}** · {GROUP}", "",
             "Deep Learning for Engineering · Aalborg University · 2026", "",
             "---", ""]
    for title, (text, _) in answers.items():
        lines += [f"## {title}", "", text.strip(), ""]
    report = "\n".join(lines)

    os.makedirs(core.OUTPUT_DIR, exist_ok=True)
    out = os.path.join(core.OUTPUT_DIR, "Ex06_report.md")
    with open(out, "w", encoding="utf-8") as fh:
        fh.write(report)
    print("wrote", out)
    print(f"{sum(len(t.split()) for t, _ in answers.values())} words total")

### The report as a PDF

Moodle shows a PDF inline and a `.md` only as a download, so the cell below
converts the report you just wrote into a PDF (with the figure, if one was
saved) and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex06_report.md into Ex06_report.pdf, with any figure
# saved as Ex06_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md = open("Ex06_report.md", encoding="utf-8").read()
figs = sorted(glob.glob("Ex06_report*.png"))
if figs:
    imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
    i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
    md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

html = markdown.markdown(md, extensions=["fenced_code", "tables"])
css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                       f"<body>{html}</body></html>", base_url=".").write_pdf("Ex06_report.pdf")
print("written Ex06_report.pdf", f"with {len(figs)} figure(s)" if figs else "")
try:
    from google.colab import files
    files.download("Ex06_report.pdf")
except ImportError:
    pass


## 4 · What Ex_06 was for, and where Part 1 ends

Four notebooks, four claims you can now defend with your own measurements:

* a loss is a noise assumption written down, not a preference;
* an optimiser is chosen for where in the landscape you are, which is why Part 2
  uses two of them in sequence;
* a trained model is an asset that transfers to a related problem, and the
  learning rate decides whether it survives the transfer;
* a deployed model is judged on three numbers, and improving one at the expense
  of the others is not an improvement.

**Part 1 ends here.** From L7 the physics enters the loss and the network stops
being fitted to data alone. Nothing about the machinery changes — the same
`tanh` networks, the same Adam-then-L-BFGS, the same insistence on a held-out
comparison. What changes is where the constraint comes from.

One thing to carry forward: in Part 2 you will frequently have no ground truth
to hold out. The residual will be all you can measure, and notebook 02 has
already shown you that a small residual and a correct answer are not the same
claim.